In [1]:
import pandas as pd
import glob;
import os;
from scipy.spatial import cKDTree
from collections import defaultdict

def merge_data(bushfire_df, weather_df: pd.DataFrame):
    print("Converting dates...")
    bushfire_df['discovery_date'] = pd.to_datetime(bushfire_df['discovery_date']).dt.date
    weather_df['date'] = pd.to_datetime(weather_df['date']).dt.date

    print("Grouping weather data...")
    weather_grouped = weather_df.groupby(['date', 'state'])

    print("Processing bushfires...")
    results = []
    total_fires = len(bushfire_df)
    missing_data_log = defaultdict(int)

    for i, (idx, fire) in enumerate(bushfire_df.iterrows()):
        if i % 1000 == 0:
            print(f"Processing fire {i+1}/{total_fires}")

        try:
            day_state_weather = weather_grouped.get_group((fire['discovery_date'], fire['state']))
            
            if not day_state_weather.empty:
                tree = cKDTree(day_state_weather[['latitude', 'longitude']])
                distance, index = tree.query([fire['latitude'], fire['longitude']])
                nearest_station = day_state_weather.iloc[index]
                
                results.append({
                    **fire.to_dict(),
                    't_min': nearest_station['t_min'],
                    't_max': nearest_station['t_max'],
                    'elevation': nearest_station['elevation']
                })
            else:
                missing_data_log[(fire['discovery_date'], fire['state'])] += 1
                results.append(fire.to_dict())
        except KeyError:
            missing_data_log[(fire['discovery_date'], fire['state'])] += 1
            results.append(fire.to_dict())

    print("Creating merged DataFrame...")
    merged_df = pd.DataFrame(results)

    print("\nMissing Weather Data Summary:")
    for (date, state), count in missing_data_log.items():
        print(f"Date: {date}, State: {state}, Missing Entries: {count}")

    return merged_df


In [2]:

# Let's load our weather and bushfire data:
weather_files = glob.glob('./../output-final/*_weather_data.csv')  # This requires the weather dataset which we aren't able to provide due to it's large size and time it takes to clean.
bushfire_df = pd.read_csv('./datasets/cleaned-bushfire-data.csv')


In [ ]:
merged_dfs = []

# Loop through each weather file
for weather_file in weather_files:
    year = os.path.basename(weather_file).split('_')[0]  # Extract year from filename
    print(f"Processing weather data for year {year}")
    
    # Read weather data for the current year
    weather_df = pd.read_csv(weather_file)
    
    # Merge bushfire data with weather data for this year
    merged_df = merge_data(bushfire_df, weather_df)
    
    # Append the result to our list
    merged_dfs.append(merged_df)

# Concatenate all merged dataframes
final_merged_df = pd.concat(merged_dfs, ignore_index=True)

# Remove any potential duplicates
final_merged_df = final_merged_df.drop_duplicates()

# Let's save the final merged dataset
final_merged_df.to_csv('~/Desktop/merged-data.csv', index=False)